# ATML PA3 — Task 1 on Kaggle (2 × T4)

Runs the whole of Task 1 (five fine-tuning runs, seven lm-eval conditions, diagnostics, qualitative generations)
from https://github.com/Hashir-afif/ATML-PA3. Two runs train in parallel, one per GPU.

**Notebook settings (right panel):** Accelerator **GPU T4 × 2**, Internet **On**.

**How to run**
1. *Save Version → Save & Run All (Commit)*. It runs in the background (you can close the tab) for up to ~11 h,
   then saves `results/` and the checkpoints the next session needs.
2. When it finishes, check the last cell's output. If it says **TASK 1 NOT FINISHED**: *Add Input → Your Work →*
   this notebook's latest version (its output), then *Save & Run All (Commit)* again. Repeat until
   **TASK 1 ALL DONE**. Only the newest attached version is used.
3. Download `results_task1.zip` from the output tab and unzip it into the local repo root
   (it contains `results/task1/...`).

Training pauses cleanly at the session deadline with a recovery checkpoint and resumes next session from exactly
that update; evaluations only start when enough time is left.

In [ ]:
# ---- parameters ----
REPO_URL = "https://github.com/Hashir-afif/ATML-PA3.git"
BRANCH = "main"
SESSION_HOURS = 11.0   # Kaggle stops sessions at 12 h; the remaining time is reserved for saving outputs
EVAL_MIN = 150         # minutes required before starting an lm-eval pass (measured: ~132 min/condition
                       # on an 8 GB GPU with the fixed batch sizes in configs/task1.yaml -- "auto" batch
                       # sizing is NOT used because it relies on a torch/accelerate OOM-message match that
                       # is broken for torch==2.11.0+cu128; 150 adds margin for a possibly slower T4)

import os, time
os.environ["PA3_DEADLINE_EPOCH"] = str(int(time.time() + SESSION_HOURS * 3600))
os.environ["EVAL_MIN"] = str(EVAL_MIN)
os.environ["PA3_CKPT_ROOT"] = "/kaggle/tmp/checkpoints/task1"
print("deadline:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(int(os.environ["PA3_DEADLINE_EPOCH"]))))

In [ ]:
!nvidia-smi -L
!mkdir -p /kaggle/tmp && df -h /kaggle/working /kaggle/tmp
!python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.version.cuda, torch.cuda.device_count())"

In [ ]:
%cd /kaggle/working
!rm -rf ATML-PA3 && git clone --depth 1 -b {BRANCH} {REPO_URL}
%cd /kaggle/working/ATML-PA3
!git log -1 --oneline

In [ ]:
!pip install -q -r kaggle/requirements-kaggle.txt
# prefetch everything Task 1 needs once, before the two chains start (avoids two processes racing to download
# the same files, and avoids Task 2/3 assets this notebook never uses): Qwen3-0.6B, MetaMathQA, GSM8K,
# first_plot_questions.yaml, and the lm-eval task datasets (GSM8K-CoT, HellaSwag, BoolQ, ARC-Easy).
!python -m scripts.prefetch --only task1
!python -c "import transformers, lm_eval, torch; print('transformers', transformers.__version__, '| lm_eval', lm_eval.__version__, '| torch', torch.__version__)"

In [ ]:
# restore results and checkpoints from the previous session's output, if one is attached as an input
!python kaggle/carry.py import
!python kaggle/carry.py status

In [ ]:
# main run: both GPU chains, then diagnostics / annotation export / summary (progress lines stream below;
# per-step logs are in results/_overnight/logs/)
!bash kaggle/run_task1_kaggle.sh

In [ ]:
# save what the next session needs, and package the results for download
!python kaggle/carry.py export
!rm -f /kaggle/working/results_task1.zip && zip -qr /kaggle/working/results_task1.zip results/task1 results/_overnight
!ls -la /kaggle/working && du -sh /kaggle/working/carry 2>/dev/null
!python kaggle/carry.py status